# SmartEV — Simulation Analysis
Run cells top to bottom on first load. After that, individual chart cells can be re-run independently.

## Setup

In [1]:
import tomllib
import polars as pl
import altair as alt
from pathlib import Path

from template.models import station_snapshot, charger_snapshot
from template.analysis import charger_analysis, station_analysis

alt.data_transformers.enable('vegafusion')

CONFIG_PATH = Path('config.toml')
with open(CONFIG_PATH, 'rb') as f:
    toml = tomllib.load(f)

perkuet_dir = (CONFIG_PATH.parent / toml['paths']['perkuet_dir']).resolve()
runs = sorted([p for p in perkuet_dir.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
run_dir = runs[-1]
print(f'Run: {run_dir.name}')

Run: b0275d54-c43d-4362-8c81-31b66948c586


In [2]:
stations = station_snapshot.load(run_dir / 'StationSnapshotMetric.parquet')
chargers = charger_snapshot.load(run_dir / 'ChargerSnapshotMetric.parquet')

# Convert SimTime from seconds to hours for readable axis labels
stations = stations.with_columns((pl.col('SimTime') / 3600).alias('SimHour'))
chargers = chargers.with_columns((pl.col('SimTime') / 3600).alias('SimHour'))

print(f'Stations: {stations.shape}, Chargers: {chargers.shape}')

Stations: (15000, 10), Chargers: (90000, 10)


## Price

In [ ]:
# Distribution of mean price per station
price = station_analysis.price_stats(stations).to_pandas()

alt.Chart(price).mark_bar(color='#2196f3', opacity=0.85).encode(
    alt.X('MeanPrice:Q', bin=alt.Bin(maxbins=40), title='Mean Price (DKK/kWh)'),
    alt.Y('count()', title='Number of Stations'),
    tooltip=[
        alt.Tooltip('MeanPrice:Q', bin=alt.Bin(maxbins=40), title='Price Range'),
        alt.Tooltip('count()', title='Stations'),
    ]
).properties(
    title='Distribution of Mean Energy Price Across Stations',
    width=700, height=350
).interactive()

alt.Chart(...)

In [4]:
# Price spread per station: how much does price vary over sim time?
price['PriceSpread'] = price['MaxPrice'] - price['MinPrice']

alt.Chart(price).mark_bar(color='#ff9800', opacity=0.85).encode(
    alt.X('PriceSpread:Q', bin=alt.Bin(maxbins=40), title='Price Spread (DKK/kWh)'),
    alt.Y('count()', title='Number of Stations'),
    tooltip=[
        alt.Tooltip('PriceSpread:Q', bin=alt.Bin(maxbins=40), title='Spread'),
        alt.Tooltip('count()', title='Stations'),
    ]
).properties(
    title='Price Spread per Station (Max − Min across snapshots)',
    width=700, height=350
).interactive()

alt.Chart(...)

In [5]:
# Price over sim time: median + IQR band across all stations
price_time = (
    stations
    .group_by('SimHour')
    .agg(
        pl.col('Price').quantile(0.25).alias('P25'),
        pl.col('Price').quantile(0.50).alias('P50'),
        pl.col('Price').quantile(0.75).alias('P75'),
    )
    .sort('SimHour')
    .to_pandas()
)

band = alt.Chart(price_time).mark_area(opacity=0.2, color='#2196f3').encode(
    x=alt.X('SimHour:Q', title='Simulation Time (hours)'),
    y=alt.Y('P25:Q', title='Price (DKK/kWh)'),
    y2='P75:Q',
)
line = alt.Chart(price_time).mark_line(color='#2196f3', strokeWidth=2).encode(
    x='SimHour:Q',
    y='P50:Q',
    tooltip=[
        alt.Tooltip('SimHour:Q', title='Hour'),
        alt.Tooltip('P50:Q', title='Median Price', format='.3f'),
        alt.Tooltip('P25:Q', title='P25', format='.3f'),
        alt.Tooltip('P75:Q', title='P75', format='.3f'),
    ]
)

(band + line).properties(
    title='Energy Price Over Simulation Time (median ± IQR across all stations)',
    width=700, height=350
).interactive()

alt.LayerChart(...)

## Queue Size

In [6]:
# Total queue size across all stations over sim time
queue_time = (
    stations
    .group_by('SimHour')
    .agg(pl.col('TotalQueueSize').sum().alias('TotalQueued'))
    .sort('SimHour')
    .to_pandas()
)

alt.Chart(queue_time).mark_line(point=True, color='#e91e63', strokeWidth=2).encode(
    x=alt.X('SimHour:Q', title='Simulation Time (hours)'),
    y=alt.Y('TotalQueued:Q', title='Total EVs Queued'),
    tooltip=[
        alt.Tooltip('SimHour:Q', title='Hour'),
        alt.Tooltip('TotalQueued:Q', title='EVs Queued'),
    ]
).properties(
    title='Total Queue Size Across All Stations Over Time',
    width=700, height=350
).interactive()

alt.Chart(...)

In [7]:
# Distribution of P90 queue size per station
queue_p90 = station_analysis.queue_size_percentiles(stations).to_pandas()

alt.Chart(queue_p90).mark_bar(color='#e91e63', opacity=0.85).encode(
    alt.X('P90:Q', bin=alt.Bin(maxbins=30), title='P90 Queue Size'),
    alt.Y('count()', title='Number of Stations'),
    tooltip=[
        alt.Tooltip('P90:Q', bin=alt.Bin(maxbins=30), title='P90 Queue'),
        alt.Tooltip('count()', title='Stations'),
    ]
).properties(
    title='Distribution of P90 Queue Size per Station',
    width=700, height=350
).interactive()

alt.Chart(...)

## Charger Utilization

In [8]:
# Utilization over sim time — dual vs single, median + IQR band
util_time = (
    chargers
    .with_columns(
        pl.when(
            pl.col('IsDual')
            .cast(pl.Utf8)
            .str.to_lowercase()
            .is_in(['true', 'dual', '1'])
        )
        .then(pl.lit('Dual'))
        .otherwise(pl.lit('Single'))
        .alias('ChargerType')
    )
    .group_by('SimHour', 'ChargerType')
    .agg(
        pl.col('Utilization').quantile(0.25).alias('P25'),
        pl.col('Utilization').quantile(0.50).alias('P50'),
        pl.col('Utilization').quantile(0.75).alias('P75'),
    )
    .sort('SimHour')
    .to_pandas()
)

color_scale = alt.Scale(domain=['Dual', 'Single'], range=['#9c27b0', '#4caf50'])

band = alt.Chart(util_time).mark_area(opacity=0.15).encode(
    x=alt.X('SimHour:Q', title='Simulation Time (hours)'),
    y=alt.Y('P25:Q', title='Utilization'),
    y2='P75:Q',
    color=alt.Color('ChargerType:N', scale=color_scale),
)
line = alt.Chart(util_time).mark_line(strokeWidth=2).encode(
    x='SimHour:Q',
    y='P50:Q',
    color=alt.Color('ChargerType:N', scale=color_scale, legend=alt.Legend(title='Charger Type')),
    tooltip=[
        alt.Tooltip('SimHour:Q', title='Hour'),
        alt.Tooltip('ChargerType:N', title='Type'),
        alt.Tooltip('P50:Q', title='Median Utilization', format='.4f'),
    ]
)

(band + line).properties(
    title='Charger Utilization Over Time — Dual vs Single (median ± IQR)',
    width=700, height=350
).interactive()

alt.LayerChart(...)

In [9]:
# Distribution of P90 utilization per charger, dual vs single overlaid
util_p90 = (
    charger_analysis.utilization_percentiles(chargers)
    .with_columns(
        pl.when(
            pl.col('IsDual')
            .cast(pl.Utf8)
            .str.to_lowercase()
            .is_in(['true', 'dual', '1'])
        )
        .then(pl.lit('Dual'))
        .otherwise(pl.lit('Single'))
        .alias('ChargerType')
    )
    .to_pandas()
)

alt.Chart(util_p90).mark_bar(opacity=0.75).encode(
    alt.X('P90:Q', bin=alt.Bin(maxbins=40), title='P90 Utilization'),
    alt.Y('count()', title='Number of Chargers', stack=None),
    alt.Color('ChargerType:N', scale=color_scale, legend=alt.Legend(title='Charger Type')),
    tooltip=[
        alt.Tooltip('P90:Q', bin=alt.Bin(maxbins=40), title='P90 Utilization'),
        alt.Tooltip('count()', title='Chargers'),
        alt.Tooltip('ChargerType:N', title='Type'),
    ]
).properties(
    title='Distribution of P90 Utilization — Dual vs Single Chargers',
    width=700, height=350
).interactive()

alt.Chart(...)

## Charger Activity

In [10]:
# Fraction of all chargers active at each snapshot
activity_time = (
    chargers
    .group_by('SimHour')
    .agg(
        ((pl.col('Utilization') > 0).sum() / pl.col('ChargerId').count()).alias('ActiveFraction')
    )
    .sort('SimHour')
    .to_pandas()
)

alt.Chart(activity_time).mark_line(point=True, color='#ff9800', strokeWidth=2).encode(
    x=alt.X('SimHour:Q', title='Simulation Time (hours)'),
    y=alt.Y('ActiveFraction:Q', title='Fraction of Active Chargers', axis=alt.Axis(format='%')),
    tooltip=[
        alt.Tooltip('SimHour:Q', title='Hour'),
        alt.Tooltip('ActiveFraction:Q', title='Active Fraction', format='.2%'),
    ]
).properties(
    title='Fraction of Active Chargers Over Simulation Time',
    width=700, height=350
).interactive()

alt.Chart(...)

In [11]:
# Station size vs mean utilization — do bigger stations utilise better?
station_util = station_analysis.utilization_percentiles(chargers)
station_size = stations.group_by('StationId').agg(pl.col('TotalChargers').first())

scatter_df = station_util.join(station_size, on='StationId').to_pandas()

alt.Chart(scatter_df).mark_circle(opacity=0.4, size=40, color='#2196f3').encode(
    x=alt.X('TotalChargers:Q', title='Number of Chargers at Station'),
    y=alt.Y('Mean:Q', title='Mean Utilization'),
    tooltip=[
        alt.Tooltip('StationId:O', title='Station'),
        alt.Tooltip('TotalChargers:Q', title='Chargers'),
        alt.Tooltip('Mean:Q', title='Mean Utilization', format='.4f'),
        alt.Tooltip('P90:Q', title='P90 Utilization', format='.4f'),
    ]
).properties(
    title='Station Size vs Mean Utilization',
    width=700, height=400
).interactive()

alt.Chart(...)